#### IMPORTS Y CONFIGURACIONES

In [ ]:
import requests
import re
import time
import os
import psycopg2

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "https://books.toscrape.com/"

OPENLIBRARY_SEARCH = "https://openlibrary.org/search.json"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
WIKIDATA_ENTITY = "https://www.wikidata.org/wiki/Special:EntityData/{}.json"

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

cache_autores = {}

REQUEST_TIMEOUT = 10

#### CONEXION POSTGRESQL + SCHEMA

In [20]:
def conectar_db():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD")
    )


conn = conectar_db()
cur = conn.cursor()

with open("schema.sql", "r", encoding="utf-8") as f:
    cur.execute(f.read())

conn.commit()

print("Base de datos lista.")

Base de datos lista.


#### OBTENER CATEGORIAS

In [11]:
response = session.get(BASE_URL)
soup = BeautifulSoup(response.text, "html.parser")

categorias = [
    {
        "name": item.text.strip(),
        "url": urljoin(BASE_URL, item["href"])
    }
    
    for item in soup.select(".side_categories ul li ul li a")
]

print("Categorías encontradas:", len(categorias))

for categoria in categorias:
    print(categoria)

Categorías encontradas: 50
{'name': 'Travel', 'url': 'https://books.toscrape.com/catalogue/category/books/travel_2/index.html'}
{'name': 'Mystery', 'url': 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html'}
{'name': 'Historical Fiction', 'url': 'https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html'}
{'name': 'Sequential Art', 'url': 'https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html'}
{'name': 'Classics', 'url': 'https://books.toscrape.com/catalogue/category/books/classics_6/index.html'}
{'name': 'Philosophy', 'url': 'https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html'}
{'name': 'Romance', 'url': 'https://books.toscrape.com/catalogue/category/books/romance_8/index.html'}
{'name': 'Womens Fiction', 'url': 'https://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html'}
{'name': 'Fiction', 'url': 'https://books.toscrape.com/catalogue/category/books/fiction_10/in

#### FUNCIONES SCRAPING LIBRO

In [ ]:
def obtener_links_categoria(url_categoria):
    pagina = 1
    links = []

    while True:

        if pagina == 1:
            url = url_categoria
        else:
            url = url_categoria.replace("index.html", f"page-{pagina}.html")

        response = session.get(url)

        if response.status_code == 404:
            break

        soup = BeautifulSoup(response.text, "html.parser")
        libros = soup.select("article.product_pod h3 a")

        if not libros:
            break

        for libro in libros:
            links.append(urljoin(url, libro["href"]))

        pagina += 1

    return links


def scrape_libro(url):

    response = session.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    title = soup.find("h1").text.strip()

    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

    clases = soup.find("p", class_="star-rating")["class"]
    rating = next((rating_map[c] for c in clases if c in rating_map), 0)
    
    desc = soup.select_one("#product_description + p")
    description = desc.text.strip() if desc else None

    img = soup.select_one(".item.active img")
    image_url = urljoin(url, img["src"]) if img else None

    tabla = {}

    for row in soup.select("table tr"):
        tabla[row.th.text.strip()] = row.td.text.strip()

    stock_txt = tabla.get("Availability", "")
    m = re.search(r"(\d+)", stock_txt)
    stock = int(m.group(1)) if m else 0

    return {
        "title": title,
        "rating": rating,
        "description": description,
        "upc": tabla.get("UPC"),
        "product_type": tabla.get("Product Type"),
        "price_excl_tax": float(tabla.get("Price (excl. tax)", "0").replace("£","").replace("Â","").strip()),
        "price_incl_tax": float(tabla.get("Price (incl. tax)", "0").replace("£","").replace("Â","").strip()),
        "tax": float(tabla.get("Tax", "0").replace("£","").replace("Â","").strip()),
        "stock": stock,
        "reviews": int(tabla.get("Number of reviews", 0)),
        "image_url": image_url,
        "product_url": url
    }

#### FUNCIONES APIs AUTOR

In [ ]:

def safe_get(url, params=None, retries=3):
    for intento in range(retries):
        try:
            r = session.get(url, params=params, timeout=REQUEST_TIMEOUT)

            if r.status_code == 404:
                print(f"404 -> {url}")
                return None
            
            if r.status_code == 429:
                espera = 2 ** intento
                print(f"Rate limit -> esperando {espera}s")
                time.sleep(espera)
                continue
            
            r.raise_for_status()
            
            return r
        
        except requests.exceptions.Timeout:
            print(f"Timeout -> {url}")
        
        except requests.exceptions.RequestException as e:
            print(f"HTTP error -> {e}")
        
        time.sleep(1)
    
    return None

def buscar_autor(titulo):
    if titulo in cache_autores:
        return cache_autores[titulo]

    resultado = {
        "author_name": None,
        "birth_year": None,
        "country": None,
        "total_known_works": None,
        "external_api_id": None,
        "api_source": None
    }

    r = safe_get(OPENLIBRARY_SEARCH, params={"title": titulo, "limit": 1})

    if r:
        docs = r.json().get("docs", [])
    
        if docs:
            doc = docs[0]
            resultado["author_name"] = doc.get("author_name", [None])[0]
            author_key = doc.get("author_key", [None])[0]
    
            if author_key:
                r2 = safe_get(f"https://openlibrary.org/authors/{author_key}/works.json")
    
                if r2:
                    resultado["total_known_works"] = r2.json().get("size")

    if not resultado["author_name"]:
        print(f"Autor no encontrado para título: {titulo}")
        cache_autores[titulo] = resultado
        return resultado

    r3 = safe_get(WIKIDATA_API, params={
        "action":"wbsearchentities","search":resultado["author_name"],
        "language":"en","format":"json"
    })
    if r3:
        encontrados = r3.json().get("search", [])
        if encontrados:
            qid = encontrados[0]["id"]
            resultado["external_api_id"] = qid
            r4 = safe_get(WIKIDATA_ENTITY.format(qid))

            if r4:
                entity = r4.json()["entities"][qid]
                claims = entity.get("claims", {})
            
                if "P569" in claims:
                    fecha = claims["P569"][0]["mainsnak"]["datavalue"]["value"]["time"]
                    resultado["birth_year"] = int(fecha[1:5])
            
                if "P27" in claims:
                    country_id = claims["P27"][0]["mainsnak"]["datavalue"]["value"]["id"]
                    r5 = safe_get(WIKIDATA_ENTITY.format(country_id))
            
                    if r5:
                        labels = r5.json()["entities"][country_id]["labels"]
            
                        if "en" in labels:
                            resultado["country"] = labels["en"]["value"]

    resultado["api_source"] = "openlibrary + wikidata" if resultado["external_api_id"] else "openlibrary"
    cache_autores[titulo] = resultado
    return resultado


#### INSERTS SQL

In [14]:
def insertar_categoria(nombre):
    cur.execute("""
        INSERT INTO categories(category_name)
        VALUES (%s)
        ON CONFLICT(category_name)
        DO UPDATE SET category_name=EXCLUDED.category_name
        RETURNING category_id
    """, (nombre,))
    return cur.fetchone()[0]


def insertar_autor(a):

    if not a["author_name"]:
        return None

    cur.execute("""
        INSERT INTO authors(
            author_name,birth_year,country,
            external_api_id,total_known_works,api_source
        )
        VALUES(%s,%s,%s,%s,%s,%s)
        ON CONFLICT(author_name)
        DO UPDATE SET
            birth_year=EXCLUDED.birth_year,
            country=EXCLUDED.country,
            external_api_id=EXCLUDED.external_api_id,
            total_known_works=EXCLUDED.total_known_works,
            api_source=EXCLUDED.api_source
        RETURNING author_id
    """, (
        a["author_name"],
        a["birth_year"],
        a["country"],
        a["external_api_id"],
        a["total_known_works"],
        a["api_source"]
    ))

    return cur.fetchone()[0]


def insertar_libro(libro, category_id):

    cur.execute("""
        INSERT INTO books(
            title,category_id,product_description,rating,
            price_excl_tax,price_incl_tax,tax,stock,
            upc,product_type,number_reviews,url_image,url_product
        )
        VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT(upc)
        DO UPDATE SET title=EXCLUDED.title
        RETURNING book_id
    """, (
        libro["title"],
        category_id,
        libro["description"],
        libro["rating"],
        libro["price_excl_tax"],
        libro["price_incl_tax"],
        libro["tax"],
        libro["stock"],
        libro["upc"],
        libro["product_type"],
        libro["reviews"],
        libro["image_url"],
        libro["product_url"]
    ))

    return cur.fetchone()[0]


def relacionar(book_id, author_id):

    if not author_id:
        return

    cur.execute("""
        INSERT INTO book_author(book_id, author_id)
        VALUES(%s,%s)
        ON CONFLICT(book_id, author_id) DO NOTHING
    """, (book_id, author_id))

#### EJECUCION TOTAL

In [ ]:
inicio = time.time()

insertados = 0
errores = 0

for n, categoria in enumerate(categorias, 1):

    print("=" * 70)
    print(f"CATEGORÍA {n}/{len(categorias)} -> {categoria['name']}")
    print("=" * 70)

    category_id = insertar_categoria(categoria["name"])

    links = obtener_links_categoria(categoria["url"])

    for i, link in enumerate(links, 1):

        try:
            libro = scrape_libro(link)
            autor = buscar_autor(libro["title"])

            book_id = insertar_libro(libro, category_id)
            author_id = insertar_autor(autor)

            relacionar(book_id, author_id)

            conn.commit()

            insertados += 1

            print(f"[{i}/{len(links)}] OK -> {libro['title']}")

        except Exception as e:
            conn.rollback()
            errores += 1
            print("ERROR:", e)

fin = time.time()

print("\nProceso finalizado")
print("Insertados:", insertados)
print("Errores:", errores)
print("Tiempo:", round(fin - inicio, 2), "segundos")

CATEGORÍA 1/50 -> Travel
Rate limit -> esperando 1s
Rate limit -> esperando 2s
Rate limit -> esperando 4s
[1/11] OK -> It's Only the Himalayas
Autor no encontrado para título: Full Moon over Noahâs Ark: An Odyssey to Mount Ararat and Beyond
[2/11] OK -> Full Moon over Noahâs Ark: An Odyssey to Mount Ararat and Beyond
Autor no encontrado para título: See America: A Celebration of Our National Parks & Treasured Sites
[3/11] OK -> See America: A Celebration of Our National Parks & Treasured Sites
Autor no encontrado para título: Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel
[4/11] OK -> Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel
Rate limit -> esperando 1s
Rate limit -> esperando 2s
Rate limit -> esperando 4s
[5/11] OK -> Under the Tuscan Sun
Rate limit -> esperando 1s
Rate limit -> esperando 2s
Rate limit -> esperando 4s
[6/11] OK -> A Summer In Europe
Rate limit -> esperando 1s
Rate limit -> esperando 2s
Rate limit -> esperando 4s
[7/11]

## CONSULTAS SQL ÚTILES (mínimo 5)

In [24]:

consultas = {
"Libros baratos rating": '''
SELECT title, rating, price_incl_tax
FROM books
WHERE rating > 3 AND price_incl_tax < 12
ORDER BY rating DESC, price_incl_tax ASC;''',

"Peor autor promedio": '''
SELECT a.author_name, ROUND(AVG(b.rating),2) promedio, COUNT(*) libros
FROM authors a
JOIN book_author ba ON ba.author_id=a.author_id
JOIN books b ON b.book_id=ba.book_id
GROUP BY a.author_name
HAVING COUNT(*)>=5
ORDER BY promedio ASC
LIMIT 1;''',

"Categoria con mayor precio primedio": '''
SELECT c.category_name, ROUND(AVG(b.price_incl_tax),2) promedio
FROM categories c JOIN books b ON b.category_id=c.category_id
GROUP BY c.category_name
ORDER BY promedio DESC LIMIT 1;''',

"Top 5 autores con mas libros": '''
SELECT a.author_name, COUNT(*) libros
FROM authors a JOIN book_author ba ON ba.author_id=a.author_id
GROUP BY a.author_name
ORDER BY libros DESC LIMIT 5;''',

"Pais que produce mas libros con rating > 3": '''
SELECT a.country, COUNT(*) libros
FROM books b
JOIN book_author ba ON ba.book_id=b.book_id
JOIN authors a ON a.author_id=ba.author_id
WHERE b.rating > 3 AND a.country IS NOT NULL
GROUP BY a.country
ORDER BY libros DESC LIMIT 1;'''
}

for nombre,q in consultas.items():
    print("\n", "="*60, "\n", nombre)
    cur.execute(q)
    print(cur.fetchall())



 Libros baratos rating
[('An Abundance of Katherines', 5, Decimal('10.00')), ('Greek Mythic History', 5, Decimal('10.23')), ('The Power Greens Cookbook: 140 Delicious Superfood Recipes', 5, Decimal('11.05')), ('Dear Mr. Knightley', 5, Decimal('11.21')), ('The Darkest Corners', 5, Decimal('11.33')), ('Naturally Lean: 125 Nourishing Gluten-Free, Plant-Based Recipes--All Under 300 Calories', 5, Decimal('11.38')), ('Fruits Basket, Vol. 2 (Fruits Basket #2)', 5, Decimal('11.64')), ('Old School (Diary of a Wimpy Kid #10)', 5, Decimal('11.83')), ('Superman Vol. 1: Before Truth (Superman by Gene Luen Yang #1)', 5, Decimal('11.89')), ('The Origin of Species', 4, Decimal('10.01')), ('History of Beauty', 4, Decimal('10.29')), ('NaNo What Now? Finding your editing process, revising your NaNoWriMo book and building a writing career through publishing and beyond', 4, Decimal('10.41')), ('I Am Pilgrim (Pilgrim #1)', 4, Decimal('10.60')), ('Green Eggs and Ham (Beginner Books B-16)', 4, Decimal('10.79

## INDEXACIÓN Y PERFORMANCE

In [21]:
consulta = '''
SELECT a.country,
       AVG(f.price_incl_tax) AS avg_price
FROM (
    SELECT book_id, price_incl_tax
    FROM books
    WHERE rating >= 3
) f
JOIN book_author ba ON ba.book_id = f.book_id
JOIN authors a ON a.author_id = ba.author_id
GROUP BY a.country
ORDER BY avg_price DESC;
'''

inicio = time.perf_counter()

cur.execute(consulta)
resultado_sin_idx = cur.fetchall()

sin_idx = time.perf_counter() - inicio

cur.execute("""
CREATE INDEX IF NOT EXISTS idx_books_rating_book_price
ON books (rating, book_id, price_incl_tax);
""")
conn.commit()

inicio = time.perf_counter()

cur.execute(consulta)
resultado_con_idx = cur.fetchall()

con_idx = time.perf_counter() - inicio

print("\nRESULTADO CONSULTA:")
for fila in resultado_con_idx:
    print(fila)

print("\nTiempo sin índice:", sin_idx)
print("Tiempo con índice:", con_idx)

if sin_idx > 0:
    mejora = round((sin_idx - con_idx) / sin_idx * 100, 2)
else:
    mejora = 0

print("Mejora:", mejora, "%")


RESULTADO CONSULTA:
('Nazi Germany', Decimal('59.9000000000000000'))
('Kingdom of the Netherlands', Decimal('55.9100000000000000'))
('Ireland', Decimal('45.5100000000000000'))
('Spain', Decimal('43.0100000000000000'))
('United Kingdom', Decimal('38.5847619047619048'))
('United States', Decimal('36.1642222222222222'))
('Australia', Decimal('35.9533333333333333'))
('German Empire', Decimal('34.2200000000000000'))
(None, Decimal('34.2136752136752137'))
('Qi', Decimal('33.3400000000000000'))
('Ionian League', Decimal('29.6400000000000000'))
('Austria', Decimal('29.4800000000000000'))
('Canada', Decimal('29.2620000000000000'))
('France', Decimal('29.0400000000000000'))
('Russian Empire', Decimal('26.5800000000000000'))
('Germany', Decimal('24.7300000000000000'))
('Iran', Decimal('24.7000000000000000'))
('United Kingdom of Great Britain and Ireland', Decimal('24.4950000000000000'))
('Kingdom of England', Decimal('24.2750000000000000'))
('Italy', Decimal('20.4450000000000000'))
('Switzerland

#### CERRAR CONEXION

In [18]:
cur.close()
conn.close()

print("Conexión cerrada.")

Conexión cerrada.
